# Phishing Detection - Colab Runbook
Notebook nay chay duoc theo kieu **Run all** cho project phishing-detection.

No ho tro 2 mode:
- `tabular`: train MLP voi `Phishing_Legitimate_full.csv`
- `url`: train hybrid CNN + BiLSTM voi feed PhishTank + legit CSV

## 1) Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

## 2) Cau hinh bien duong dan

- `REPO_URL`: URL repo GitHub cua ban (de clone code)
- `DRIVE_DATA_DIR`: thu muc tren Drive chua cac file CSV
- `OUTPUT_DIR`: noi luu model/plots

In [ ]:
import os

REPO_URL = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'
PROJECT_DIR = '/content/phishing-detection'
DRIVE_DATA_DIR = '/content/drive/MyDrive/phishing_data'
OUTPUT_DIR = '/content/drive/MyDrive/phishing_outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)
print('DRIVE_DATA_DIR =', DRIVE_DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

## 3) Lay source code

In [ ]:
!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls -la

## 4) Cai dat dependencies

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 5) Dong bo du lieu vao data/raw
Yeu cau trong Drive:
- `Phishing_Legitimate_full.csv`
- `top-1m.csv` (chi can cho mode url)

In [ ]:
import shutil

raw_dir = os.path.join(PROJECT_DIR, 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

tabular_src = os.path.join(DRIVE_DATA_DIR, 'Phishing_Legitimate_full.csv')
top1m_src = os.path.join(DRIVE_DATA_DIR, 'top-1m.csv')

if os.path.exists(tabular_src):
    shutil.copy2(tabular_src, os.path.join(raw_dir, 'Phishing_Legitimate_full.csv'))
    print('Copied:', tabular_src)
else:
    print('Missing:', tabular_src)

if os.path.exists(top1m_src):
    shutil.copy2(top1m_src, os.path.join(raw_dir, 'top-1m.csv'))
    print('Copied:', top1m_src)
else:
    print('Optional missing for URL mode:', top1m_src)

!ls -lh data/raw

## 6) Kiem tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 7) Train/Eval mode TABULAR (khuyen nghi chay truoc)

In [ ]:
%cd {PROJECT_DIR}
!python main.py \
  --dataset-mode tabular \
  --tabular-csv data/raw/Phishing_Legitimate_full.csv \
  --label-col CLASS_LABEL \
  --epochs 20 \
  --batch-size 256 \
  --output-dir {OUTPUT_DIR}/tabular \
  --checkpoint {OUTPUT_DIR}/tabular/best_model_tabular.pt

## 8) Train/Eval mode URL (tuy chon)
Can co file `data/raw/top-1m.csv`.

In [ ]:
%cd {PROJECT_DIR}
!python main.py \
  --dataset-mode url \
  --legit-csv data/raw/top-1m.csv \
  --max-phishing 50000 \
  --max-legit 50000 \
  --epochs 5 \
  --batch-size 512 \
  --log-interval 10 \
  --output-dir {OUTPUT_DIR}/url \
  --checkpoint {OUTPUT_DIR}/url/best_model_url.pt

## 9) Xem ket qua da luu

In [ ]:
!find {OUTPUT_DIR} -maxdepth 3 -type f | sort | cat